In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train = pd.read_parquet(PROCESSED_DIR / "train_events_v1.parquet")
valid = pd.read_parquet(PROCESSED_DIR / "validation_events_v1.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_events_v1.parquet")

feature_columns = pd.read_csv(
    PROCESSED_DIR / "feature_list_v1.csv"
)["feature_name"].tolist()

TARGET = "reversal"
CATEGORICAL_FEATURES = ["sector"]
NUMERIC_FEATURES = feature_columns

print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", CATEGORICAL_FEATURES)
print("Train / validation / test:", len(train), len(valid), len(test))

Numeric features: 22
Categorical features: ['sector']
Train / validation / test: 11288 1771 1785


In [2]:
model_columns = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_train = train[model_columns].copy()
y_train = train[TARGET].copy()

X_valid = valid[model_columns].copy()
y_valid = valid[TARGET].copy()

X_test = test[model_columns].copy()
y_test = test[TARGET].copy()

print(X_train.shape, X_valid.shape, X_test.shape)

(11288, 23) (1771, 23) (1785, 23)


In [3]:
numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, NUMERIC_FEATURES),
        ("categorical", categorical_preprocessor, CATEGORICAL_FEATURES)
    ]
)

In [4]:
def evaluate_model(model, X, y, split_name, threshold=0.50):
    probabilities = model.predict_proba(X)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    return {
        "model": model.__class__.__name__,
        "split": split_name,
        "threshold": threshold,
        "accuracy": round(accuracy_score(y, predictions), 4),
        "precision": round(precision_score(y, predictions, zero_division=0), 4),
        "recall": round(recall_score(y, predictions, zero_division=0), 4),
        "f1": round(f1_score(y, predictions, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y, probabilities), 4),
        "pr_auc": round(average_precision_score(y, probabilities), 4),
    }

In [5]:
dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DummyClassifier(strategy="prior"))
    ]
)

dummy_pipeline.fit(X_train, y_train)

dummy_results = evaluate_model(
    dummy_pipeline,
    X_valid,
    y_valid,
    split_name="validation"
)

pd.DataFrame([dummy_results])

,model,split,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Pipeline,validation,0.5,0.8024,0.0,0.0,0.0,0.5,0.1976


In [6]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

logistic_pipeline.fit(X_train, y_train)

logistic_results = evaluate_model(
    logistic_pipeline,
    X_valid,
    y_valid,
    split_name="validation"
)

pd.DataFrame([logistic_results])

,model,split,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Pipeline,validation,0.5,0.7578,0.3073,0.18,0.227,0.5619,0.2646


In [7]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=8,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

rf_pipeline.fit(X_train, y_train)

rf_results = evaluate_model(
    rf_pipeline,
    X_valid,
    y_valid,
    split_name="validation"
)

pd.DataFrame([rf_results])

,model,split,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Pipeline,validation,0.5,0.7798,0.3684,0.16,0.2231,0.5986,0.2974


In [8]:
baseline_results = pd.DataFrame([
    dummy_results,
    logistic_results,
    rf_results
])

baseline_results.sort_values(
    ["pr_auc", "f1"],
    ascending=False
)

,model,split,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc
2,Pipeline,validation,0.5,0.7798,0.3684,0.16,0.2231,0.5986,0.2974
1,Pipeline,validation,0.5,0.7578,0.3073,0.18,0.2270,0.5619,0.2646
0,Pipeline,validation,0.5,0.8024,0.0000,0.00,0.0000,0.5000,0.1976


In [9]:
best_baseline = rf_pipeline  

valid_probabilities = best_baseline.predict_proba(X_valid)[:, 1]
valid_predictions = (valid_probabilities >= 0.50).astype(int)

print(classification_report(
    y_valid,
    valid_predictions,
    target_names=["No reversal", "Reversal"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_valid, valid_predictions))

              precision    recall  f1-score   support

 No reversal       0.82      0.93      0.87      1421
    Reversal       0.37      0.16      0.22       350

    accuracy                           0.78      1771
   macro avg       0.59      0.55      0.55      1771
weighted avg       0.73      0.78      0.74      1771

Confusion matrix:
[[1325   96]
 [ 294   56]]


In [10]:
threshold_rows = []

rf_valid_probabilities = rf_pipeline.predict_proba(X_valid)[:, 1]

for threshold in np.arange(0.20, 0.71, 0.05):
    predictions = (rf_valid_probabilities >= threshold).astype(int)

    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "predicted_reversals": int(predictions.sum()),
        "precision": precision_score(y_valid, predictions, zero_division=0),
        "recall": recall_score(y_valid, predictions, zero_division=0),
        "f1": f1_score(y_valid, predictions, zero_division=0),
    })

threshold_results = pd.DataFrame(threshold_rows)

threshold_results[
    ["precision", "recall", "f1"]
] = threshold_results[
    ["precision", "recall", "f1"]
].round(4)

threshold_results

,threshold,predicted_reversals,precision,recall,f1
0,0.20,1771,0.1976,1.0000,0.3300
1,0.25,1761,0.1988,1.0000,0.3316
2,0.30,1683,0.2026,0.9743,0.3355
3,0.35,1371,0.2101,0.8229,0.3347
4,0.40,819,0.2491,0.5829,0.3490
5,0.45,396,0.3005,0.3400,0.3190
6,0.50,152,0.3684,0.1600,0.2231
7,0.55,46,0.5652,0.0743,0.1313
8,0.60,2,1.0000,0.0057,0.0114
9,0.65,0,0.0000,0.0000,0.0000


In [11]:
best_threshold_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = best_threshold_row["threshold"]

print("Best validation threshold by F1:", best_threshold)
print(best_threshold_row)

Best validation threshold by F1: 0.4
threshold                0.4000
predicted_reversals    819.0000
precision                0.2491
recall                   0.5829
f1                       0.3490
Name: 4, dtype: float64


In [12]:
rf_best_predictions = (
    rf_valid_probabilities >= best_threshold
).astype(int)

print(classification_report(
    y_valid,
    rf_best_predictions,
    target_names=["No reversal", "Reversal"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_valid, rf_best_predictions))

              precision    recall  f1-score   support

 No reversal       0.85      0.57      0.68      1421
    Reversal       0.25      0.58      0.35       350

    accuracy                           0.57      1771
   macro avg       0.55      0.58      0.51      1771
weighted avg       0.73      0.57      0.61      1771

Confusion matrix:
[[806 615]
 [146 204]]


In [13]:
# Save Random Forest threshold-tuning results
threshold_results.to_csv(
    PROCESSED_DIR / "random_forest_threshold_results_v1.csv",
    index=False
)

# Save the chosen threshold and validation metrics
rf_selected_summary = pd.DataFrame([{
    "model": "Random Forest",
    "selection_metric": "validation_f1",
    "selected_threshold": float(best_threshold),
    "validation_precision": float(best_threshold_row["precision"]),
    "validation_recall": float(best_threshold_row["recall"]),
    "validation_f1": float(best_threshold_row["f1"]),
    "validation_pr_auc": float(rf_results["pr_auc"]),
    "validation_roc_auc": float(rf_results["roc_auc"])
}])

rf_selected_summary.to_csv(
    PROCESSED_DIR / "random_forest_selected_config_v1.csv",
    index=False
)

print("Saved:")
print(PROCESSED_DIR / "random_forest_threshold_results_v1.csv")
print(PROCESSED_DIR / "random_forest_selected_config_v1.csv")

rf_selected_summary

Saved:
/Users/kartiksinghai/Desktop/nse-stock-reversal-predictor/data/processed/random_forest_threshold_results_v1.csv
/Users/kartiksinghai/Desktop/nse-stock-reversal-predictor/data/processed/random_forest_selected_config_v1.csv


,model,selection_metric,selected_threshold,validation_precision,validation_recall,validation_f1,validation_pr_auc,validation_roc_auc
0,Random Forest,validation_f1,0.4,0.2491,0.5829,0.349,0.2974,0.5986
